In [9]:
import pandas as pd
import numpy as np
import networkx as nx
import random
import warnings
from sklearn.metrics import accuracy_score, v_measure_score, homogeneity_score, completeness_score
from scipy.spatial.distance import jensenshannon
from IPython.display import display

# Suppress sklearn warnings about highly fragmented classes if any remain
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.metrics.cluster")

# ---------------------------------------------------------
# 1. EVALUATION FUNCTIONS 
# ---------------------------------------------------------

def calculate_pairwise_accuracy(df_clusters, df_ratings, threshold=2.5):
    """Calculates pairwise accuracy by processing per compound."""
    accuracies = []
    
    for word, word_ratings in df_ratings.groupby('target_word'):
        word_clusters = df_clusters[df_clusters['compound'] == word]
        cluster_map = dict(zip(word_clusters['sentence_id'], word_clusters['cluster_id']))
        
        eval_df = word_ratings.copy()
        eval_df['cluster_1'] = eval_df['sentence_1_id'].map(cluster_map)
        eval_df['cluster_2'] = eval_df['sentence_2_id'].map(cluster_map)
        
        eval_df = eval_df.dropna(subset=['cluster_1', 'cluster_2'])
        
        if len(eval_df) > 0:
            eval_df['is_human_related'] = eval_df['[Bewertung]'] >= threshold
            eval_df['is_model_related'] = eval_df['cluster_1'] == eval_df['cluster_2']
            accuracies.append(accuracy_score(eval_df['is_human_related'], eval_df['is_model_related']))
            
    return np.mean(accuracies) if accuracies else 0.0


def generate_wug_gold_clusters(df_ratings, threshold_center=2.5, iterations=100):
    """
    Generates gold clusters per word using Correlation Clustering (Bansal et al., 2004).
    Averages multiple annotator judgements and optimizes via repeated Pivot heuristic.
    """
    gold_data = []
    
    for word, word_ratings in df_ratings.groupby('target_word'):
        G = nx.Graph()
        
        # 1. Average the ratings if multiple annotators rated the exact same sentence pair
        pair_weights = word_ratings.groupby(['sentence_1_id', 'sentence_2_id'])['[Bewertung]'].mean().reset_index()
        
        # Build the graph with the averaged weights
        for _, row in pair_weights.iterrows():
            u, v = row['sentence_1_id'], row['sentence_2_id']
            # Edges > 0 mean related (should cluster), < 0 mean unrelated (should separate)
            weight = row['[Bewertung]'] - threshold_center
            G.add_edge(u, v, weight=weight)
            
        nodes = list(G.nodes())
        if not nodes:
            continue
            
        best_clusters = None
        min_cost = float('inf')
        
        # 2. Run the Pivot heuristic multiple times to find the optimal cluster layout
        for _ in range(iterations):
            unclustered = set(nodes)
            current_partition = {}
            cluster_id_counter = 0
            
            while unclustered:
                pivot = random.choice(list(unclustered))
                current_cluster = {pivot}
                
                for neighbor in G.neighbors(pivot):
                    if neighbor in unclustered and G[pivot][neighbor]['weight'] > 0:
                        current_cluster.add(neighbor)
                        
                for node in current_cluster:
                    current_partition[node] = cluster_id_counter
                    unclustered.remove(node)
                cluster_id_counter += 1
                
            # 3. Calculate Correlation Clustering Cost for this run
            cost = 0
            for u, v, data in G.edges(data=True):
                w = data['weight']
                is_same_cluster = (current_partition[u] == current_partition[v])
                
                if is_same_cluster and w < 0:
                    cost += abs(w)  # Penalty for grouping unrelated sentences
                elif not is_same_cluster and w > 0:
                    cost += w       # Penalty for separating related sentences
                    
            # Keep the layout with the lowest penalty cost
            if cost < min_cost:
                min_cost = cost
                best_clusters = current_partition
                
        # Store the best configuration found for this word
        if best_clusters:
            for node, c_id in best_clusters.items():
                gold_data.append({'compound': word, 'sentence_id': node, 'gold_cluster': c_id})
            
    return pd.DataFrame(gold_data)


def evaluate_with_v_measure(df_pred, gold_df):
    """Evaluates V-measure per compound and averages the results."""
    homogenities, completenesses, v_measures = [], [], []
    
    for word in gold_df['compound'].unique():
        word_gold = gold_df[gold_df['compound'] == word]
        word_pred = df_pred[df_pred['compound'] == word]
        
        df_merged = pd.merge(word_gold, word_pred, on='sentence_id', how='inner')
        if len(df_merged) > 0:
            gold_labels = df_merged['gold_cluster']
            pred_labels = df_merged['cluster_id']
            
            homogenities.append(homogeneity_score(gold_labels, pred_labels))
            completenesses.append(completeness_score(gold_labels, pred_labels))
            v_measures.append(v_measure_score(gold_labels, pred_labels))
            
    return (np.mean(homogenities) if homogenities else 0.0, 
            np.mean(completenesses) if completenesses else 0.0, 
            np.mean(v_measures) if v_measures else 0.0)


def calculate_jsd(df):
    """Calculates JSD between early and late usage per compound."""
    jsd_values = []
    
    for word, group in df.groupby('compound'):
        all_clusters = sorted(group['cluster_id'].unique())
        
        early_df = group[group['year'] == 'early']
        late_df = group[group['year'] == 'late']
        
        if len(early_df) == 0 or len(late_df) == 0:
            continue

        early_counts = early_df['cluster_id'].value_counts().reindex(all_clusters, fill_value=0)
        late_counts = late_df['cluster_id'].value_counts().reindex(all_clusters, fill_value=0)
        
        p = early_counts.values / early_counts.sum()
        q = late_counts.values / late_counts.sum()
        
        js_distance = jensenshannon(p, q, base=2)
        jsd_values.append(js_distance ** 2)
        
    return np.mean(jsd_values) if jsd_values else 0.0

# ---------------------------------------------------------
# 2. DATA LOADING & COMPARISON EXECUTION
# ---------------------------------------------------------

RATINGS_FILE = 'german_annotation_result.csv'         
KMEANS_FILE = 'clustering_results.csv'     
CW_FILE = 'cw_clustering_results.csv'

try:
    print("Loading datasets...")
    df_ratings = pd.read_csv(RATINGS_FILE)
    df_kmeans = pd.read_csv(KMEANS_FILE)
    df_cw = pd.read_csv(CW_FILE)
    
    # Ensure rating column is numeric
    df_ratings['[Bewertung]'] = pd.to_numeric(df_ratings['[Bewertung]'], errors='coerce')
    df_ratings = df_ratings.dropna(subset=['[Bewertung]'])

    print("Generating robust gold clusters from human ratings (Correlation Clustering)...")
    gold_df = generate_wug_gold_clusters(df_ratings, threshold_center=2.5, iterations=100)
    
    # --- Evaluate K-Means (Baseline) ---
    print("Evaluating K-Means...")
    km_acc = calculate_pairwise_accuracy(df_kmeans, df_ratings)
    km_hom, km_comp, km_v = evaluate_with_v_measure(df_kmeans, gold_df)
    km_jsd = calculate_jsd(df_kmeans)
    
    # Initialize the results dictionary with the baseline
    results_dict = {
        "Metric": [
            "Pairwise Accuracy", 
            "Homogeneity", 
            "Completeness", 
            "V-Measure", 
            "JSD"
        ],
        "K-Means (Baseline)": [km_acc, km_hom, km_comp, km_v, km_jsd]
    }
    
    # --- Evaluate Chinese Whispers per 'k' ---
    print("Evaluating Chinese Whispers configurations...")
    # Group the CW results by the 'requested_k' parameter used in the KNN graph
    for k_val, df_cw_k in sorted(df_cw.groupby('requested_k')):
        print(f"  -> Processing CW (k={k_val})...")
        
        cw_acc = calculate_pairwise_accuracy(df_cw_k, df_ratings)
        cw_hom, cw_comp, cw_v = evaluate_with_v_measure(df_cw_k, gold_df)
        cw_jsd = calculate_jsd(df_cw_k)
        
        # Add this specific k-configuration to our results dictionary
        results_dict[f"CW (k={k_val})"] = [cw_acc, cw_hom, cw_comp, cw_v, cw_jsd]
    
    # --- Format and Display Comparison ---
    results = pd.DataFrame(results_dict)
    
    # Format all numeric columns for readability
    numeric_cols = [col for col in results.columns if col != "Metric"]
    for col in numeric_cols:
        results[col] = results[col].apply(lambda x: f"{x:.4f}")
        
    print("\n" + "="*80)
    print("                       CLUSTERING MODEL COMPARISON")
    print("="*80)
    display(results.style.hide(axis="index"))
    
except FileNotFoundError as e:
    print(f"\nError: Could not find one of the files. Please ensure the file exists.\nDetails: {e}")

Loading datasets...
Generating robust gold clusters from human ratings (Correlation Clustering)...
Evaluating K-Means...
Evaluating Chinese Whispers configurations...
  -> Processing CW (k=5)...
  -> Processing CW (k=10)...
  -> Processing CW (k=20)...

                       CLUSTERING MODEL COMPARISON


Metric,K-Means (Baseline),CW (k=5),CW (k=10),CW (k=20)
Pairwise Accuracy,0.3495,0.3248,0.3315,0.3516
Homogeneity,0.5957,0.8377,0.7252,0.5791
Completeness,0.7399,0.7848,0.7618,0.7245
V-Measure,0.6552,0.8084,0.7397,0.6360
JSD,0.1923,0.4735,0.2908,0.1610


In [ ]:
print("="*40)
print("      K-MEANS CLUSTERING RESULTS")
print("="*40)
print(f"Pairwise Accuracy : {km_acc:.4f}")
print(f"Homogeneity       : {km_hom:.4f}")
print(f"Completeness      : {km_comp:.4f}")
print(f"V-Measure         : {km_v:.4f}")
print(f"JSD               : {km_jsd:.4f}")
print("="*40)